# Check 2: Louvain Gap — Pairs Without Communities

This notebook runs **Check 2** from the Genie Test Protocol end to end.

The check asks Genie to find clusters of accounts transferring heavily among themselves — the same question a fraud analyst would ask when investigating coordinated activity. Genie returns bilateral pairs (accounts that exchange money with each other), while the actual fraud structure is a 100-account ring with dense internal transfers that Louvain resolves as a single community.

**Four phases:**

- **Phase 1** — Call Genie with the Check 2 question and display the result as a table
- **Phase 2** — Load the ground truth ring membership from the UC Volume
- **Phase 3** — Compute the ring footprint metric and determine PASS or FAIL
- **Phase 4** — Explain why SQL bilateral pairs cannot replicate Louvain community detection

**Before running:** Set your `SPACE_ID` in the configuration cell below. Run `setup/upload_and_create_tables.sh` first to load the tables and upload `ground_truth.json` to the UC Volume.

**Pass criterion:** `largest_ring_footprint <= 20` — the check passes when Genie's pairs expose at most 20 distinct accounts from any single fraud ring. The actual rings contain 100 accounts each. Genie cannot see the community boundary.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Set SPACE_ID to your Genie Space ID before running.
# Find it in Databricks under Genie → your space → the ID in the URL.

SPACE_ID = "YOUR-GENIE-SPACE-ID"   # <-- replace this

CATALOG = "graph-enriched-lakehouse"
SCHEMA  = "graph-enriched-schema"
VOLUME  = "graph-enriched-volume"

In [ ]:
from databricks.sdk import WorkspaceClient
import pandas as pd

from demo_utils import genie_caller, load_ground_truth, build_ring_lookup, check_community_pairs

w = WorkspaceClient()
print("Connected to:", w.config.host)

ask_genie = genie_caller(w, SPACE_ID)

## Phase 1 — Call Genie and Display Results

The question below asks Genie to find clusters of accounts that predominantly transfer money among themselves. This is the community detection question — the analyst is looking for coordinated activity, not individual suspicious accounts.

Genie's best SQL approximation is a bidirectional pair query: find pairs of accounts that send money to each other, ranked by how frequently they exchange transfers. The result is a list of bilateral relationships — account A and account B with 3–4 mutual transfers. If Genie attempts to group these pairs into clusters, it produces small collections of a few accounts, not the 100-account rings that Louvain resolves.

The result looks like it found something. The account pairs it surfaces are often real ring members — they do exchange money with each other. What the result cannot show is that those two accounts belong to a 100-member community with a distinct boundary separating it from adjacent rings.

In [ ]:
CHECK_2_QUESTION = (
    "Which groups of accounts are transferring money heavily among themselves? "
    "Try to identify clusters of accounts that predominantly send money to each other."
)

print("Question:", CHECK_2_QUESTION)
print()

response = ask_genie(CHECK_2_QUESTION)
print(f"Status:  {response['status']}")

if response["text"] and response["df"] is None:
    print(f"\nGenie returned a text response instead of a table:\n{response['text']}")
    raise RuntimeError(
        "Genie did not return a query result. "
        "Confirm the Genie Space is connected to the correct tables and try again."
    )

print(f"Rows returned: {len(response['df'])}")

In [ ]:
# Genie's answer — the raw result before any comparison
print("Genie returned these account pairs or clusters:\n")
display(response["df"])

In [ ]:
# The SQL Genie generated — shows what Genie actually measured
print("SQL Genie generated:\n")
print(response["sql"])

## Phase 2 — Load Ground Truth Ring Membership

The ten fraud rings each contain 100 accounts with a dense internal transfer network. `ground_truth.json` records the full ring membership — uploaded to the UC Volume by `setup/upload_and_create_tables.sh`.

Loading this file gives us the ring membership needed to evaluate how many accounts from any single ring Genie's pairs revealed.

In [ ]:
GROUND_TRUTH_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/ground_truth.json"

gt = load_ground_truth(GROUND_TRUTH_PATH)
_, whale_ids = build_ring_lookup(gt)

rings             = gt["rings"]
total_rings       = len(rings)
accounts_per_ring = len(rings[0]["account_ids"]) if rings else 0
total_fraud       = sum(len(r["account_ids"]) for r in rings)

print(f"Ground truth loaded from: {GROUND_TRUTH_PATH}")
print(f"  Rings:                {total_rings}")
print(f"  Accounts per ring:    {accounts_per_ring}")
print(f"  Total fraud accounts: {total_fraud:,}")
print(f"  Whale accounts:       {len(whale_ids):,}")
print()
print(f"Pass criterion: Genie's pairs expose <= 20 distinct accounts from any single ring.")
print(f"A passing result means Genie saw at most 20 of {accounts_per_ring} ring members.")

## Phase 3 — Compute the Ring Footprint Metric

The metric for this check is `largest_ring_footprint`: across all pairs Genie returned, for each fraud ring, count how many distinct ring accounts appear. Take the maximum across all rings.

- **Low footprint (≤ 20):** Genie surfaced only a handful of accounts from any single ring — isolated bilateral pairs, not a community. **This is a passing result.** It confirms the Louvain gap: Genie's SQL cannot find the 100-account boundary.
- **High footprint (> 20):** Genie's query exposed a large fraction of a ring — possibly via a transitive-closure CTE that chained pairs together. This would mean the community-vs-pairs gap is narrower than intended.

Note that finding same-ring pairs is expected — ring members do exchange money. What matters is the count: pairs vs. a full community.

In [ ]:
# Extract pairs from Genie's result.
# Expected columns: account_id_a, account_id_b
# If Genie returned different column names, adjust col_a and col_b below.
df = response["df"]

col_a = next(
    (c for c in df.columns if "account_id_a" in c.lower() or (c.lower().endswith("_a") and "account" in c.lower())),
    None,
)
col_b = next(
    (c for c in df.columns if "account_id_b" in c.lower() or (c.lower().endswith("_b") and "account" in c.lower())),
    None,
)

if col_a is None or col_b is None:
    # Fall back: use first two columns
    cols = df.columns.tolist()
    col_a, col_b = cols[0], cols[1]
    print(f"Note: could not detect pair columns by name. Using '{col_a}' and '{col_b}'.")
else:
    print(f"Pair columns: '{col_a}' and '{col_b}'")

pairs = list(zip(df[col_a].tolist(), df[col_b].tolist()))
print(f"Pairs extracted: {len(pairs)}")

In [ ]:
ring_lists = [r["account_ids"] for r in gt["rings"]]
result = check_community_pairs(pairs, ring_lists)

print("Ring footprint analysis:")
print(f"  Total pairs returned:   {result['total_pairs']}")
print(f"  Same-ring pairs:        {result['same_ring_pairs']}")
print(f"  Cross-ring pairs:       {result['cross_ring_pairs']}")
print(f"  Unknown pairs:          {result['unknown_pairs']}")
print(f"  Rings touched:          {result['rings_touched']}")
print()
print(f"  Largest ring footprint: {result['largest_ring_footprint']}")
print(f"  Pass criterion:         <= 20")

In [ ]:
print("=" * 62)
print("CHECK 2 SUMMARY")
print("=" * 62)
print()
print(f"Genie returned {result['total_pairs']} account pairs.")
print()
print(f"  Same-ring pairs:        {result['same_ring_pairs']}")
print(f"    Ring members do exchange money — finding same-ring pairs")
print(f"    is expected. The question is how many were revealed.")
print()
print(f"  Largest ring footprint: {result['largest_ring_footprint']} / {accounts_per_ring}")
print(f"    Genie surfaced at most {result['largest_ring_footprint']} distinct accounts")
print(f"    from any single fraud ring. Each ring has {accounts_per_ring} members.")
print()
print("-" * 62)

if result["passed"]:
    print("CHECK 2: PASS")
    print("-" * 62)
    print()
    print("Genie's pairs expose a small fraction of the ring structure.")
    print("The 100-account community boundary is invisible to this query.")
    print("This confirms the gap that Louvain closes in Check 5.")
else:
    print("CHECK 2: FAIL")
    print("-" * 62)
    print()
    print(f"largest_ring_footprint {result['largest_ring_footprint']} > 20.")
    print("Genie surfaced more ring accounts than a bilateral pair query should.")
    print("Genie may have written a transitive-closure CTE that chains pairs")
    print("together. The community-vs-pairs gap is narrower than intended.")
    print("Re-run setup/verify_fraud_patterns.py to check structural properties.")

print()
print("-" * 62)
print("Next: Check 5 (Louvain after GDS)")
print("-" * 62)
print()
print("After GDS enrichment the accounts table gains a community_id column.")
print("Genie can then answer the same question by grouping on that column —")
print("returning full 100-account communities instead of isolated pairs.")

## Phase 4 — Why SQL Bilateral Pairs Cannot Replicate Louvain

The gap confirmed by this check is not a limitation of Genie — it is a limitation of SQL itself for this class of problem.

### What Genie can do

Genie can write a bidirectional pair query: find accounts where A transfers to B and B transfers to A, sort by transfer count. It can also attempt transitive closure — a recursive CTE that chains pairs together. Both of these find ring members exchanging money. Neither can compute community boundaries.

### What Louvain does that SQL cannot

Louvain runs iterative global modularity optimization. In each iteration, every node evaluates whether moving to a neighboring community would increase the global modularity score — a measure of how much denser the within-community edges are compared to a statistical null model. The algorithm updates all assignments simultaneously and repeats until global modularity converges.

Two structural consequences follow:

1. **SQL transitive closure merges rings that share any cross-ring edge.** If ring 3 and ring 7 happen to share one background transfer, a recursive CTE chains them into a single component. Louvain's modularity objective evaluates the entire graph state at once: even with a shared edge, if the internal densities of rings 3 and 7 are high and distinct, Louvain keeps them separated because merging them would reduce global modularity.

2. **The boundary computation requires the full graph.** Louvain draws community boundaries based on every node's relationship to every other node in the graph. SQL can only evaluate local relationships — the rows it has been given. There is no SQL formulation that globally optimizes across all accounts simultaneously without materializing the full graph.

### What an analyst sees

The bilateral pairs in Phase 1 look meaningful — they are real ring members exchanging money. What the analyst cannot tell from the pairs alone is that those two accounts belong to a 100-member group with a shared boundary distinct from neighboring rings. The pairs provide no basis for deciding where the community ends. Louvain's `community_id` column provides exactly that.